##  Bronze to Silver Job

- 1. Verify the column names and standardize to snake_case
- 2. Treat the city administrative descriptors columns 
- 3. Treat the survey indicators from "Informática e Comunicação" worksheet
- 4. Treat the survey indicators from "Governanca" worksheet
- 5. Integrity Check 
    - primary key uniqueness and row count match (5,570 == 5,570)
- 6. Merge data tables
- 7. Write to silver

In [0]:
%run ../environment/env_variables

In [0]:
%run ../environment/env_functions

In [0]:
import pyspark.sql.functions as F

In [0]:
# Paths of where the final tables are going to be stored
bronze_tbl_dict = f'{catalog_name}.{bronze_schema}.munic_raw_dict'
bronze_tbl_info = f'{catalog_name}.{bronze_schema}.munic_raw_info_data'
bronze_tbl_gov = f'{catalog_name}.{bronze_schema}.munic_raw_gov_data'

silver_tbl_dict = f'{catalog_name}.{silver_schema}.munic_clean_dict'
silver_tbl_data = f'{catalog_name}.{silver_schema}.munic_clean_data'

## Step 1

In [0]:
# Load bronze tables, filtered by the survey edition year
ano_edicao = int(reference_year) if isinstance(reference_year, (int, float, str)) else 2024

df_raw_dict = spark.table(bronze_tbl_dict).filter(F.col("ano_edicao") == ano_edicao)
df_raw_info = spark.table(bronze_tbl_info).filter(F.col("ano_edicao") == ano_edicao)
df_raw_gov = spark.table(bronze_tbl_gov).filter(F.col("ano_edicao") == ano_edicao)

In [0]:
# Verify that all column names are standardized to snake_case
import re

# Define a regex expression to match snake_case
regex_expr = r'^[a-z0-9_]+$'

for name, df in [("munic_raw_dict", df_raw_dict), ("munic_raw_info", df_raw_info), ("munic_raw_gov", df_raw_gov)]:
    non_snake = [col for col in df.columns if not re.match(regex_expr, col)]

    print(f"{name}: {len(df.columns)} columns, non-snake_case: {non_snake or 'none'}")

## Step 2 

In [0]:
df_city_metainfo = df_raw_info.select("cod_munic", "desc_mun", "uf", "populacao", "faixa_populacao", "regiao")

In [0]:
df_city_metainfo.printSchema()

In [0]:
# 1. Transform the cod_munic into string 
# 2. Drop the column 'cod_munic'
# 3. Rename columns for clarity
def treat_city_metainfo(df):
    return (
        df
        .withColumn("cod_ibge", F.col("cod_munic").cast("string"))
        .drop("cod_munic")
        .withColumnsRenamed({
            "desc_mun": "nome",
            "uf": "uf_sigla",
            "populacao": "populacao_total",
        })
    )


In [0]:
# Apply the treatment to the city metainfo
df_city_metainfo = treat_city_metainfo(df_city_metainfo)
display(df_city_metainfo.limit(5))

## Step 3 - Treat the survey indicators from "Informática e Comunicação" worksheet

In [0]:
# Select only cod_munic and the MTIC indicator columns from the info table
# Drop the city descriptor and metadata columns
mtic_cols = [c for c in df_raw_info.columns if c.startswith("mtic")]
df_info_indicators = df_raw_info.select("cod_munic", *mtic_cols)

print(f"Info indicators: {len(mtic_cols)} MTIC columns")
display(df_info_indicators.limit(5))

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# Reusable function: treat IBGE MUNIC survey indicator columns.
# Works for any worksheet (Informática, Governanca, etc.) given its column prefix.
# ─────────────────────────────────────────────────────────────────────────────

def treat_survey_indicators(df, column_prefix, dict_table_name):
    """
    Classify indicator columns as binary or categorical, then apply treatment.

    Binary      → Sim=True, Não=False, non-response=NULL
    Categorical → trim whitespace, non-response=NULL, keep as string

    Parameters
    ----------
    df               : Spark DataFrame with cod_munic + indicator columns
    column_prefix    : lowercase prefix ("mtic" or "mgov")
    dict_table_name  : fully qualified dictionary table name

    Returns a dict: df, binary_cols, categorical_cols, meanings
    """
    # Standard IBGE non-response codes to convert to NULL
    NON_RESPONSE = ["Recusa", "Não informou", "Não sabe informar", "Não sabe", "-"]

    # Load dictionary meanings for this prefix (codes stored uppercase in dict)
    df_dict = spark.table(dict_table_name)
    meanings = {
        row["cod_variable"].lower(): row["meaning"]
        for row in df_dict.filter(
            F.col("cod_variable").rlike(f"^{column_prefix.upper()}")
        ).collect()
    }
    print(f"Loaded {len(meanings)} {column_prefix.upper()} meanings from dictionary")

    # Cache column list once (avoids repeated Analyze RPCs on Spark Connect)
    indicator_cols = [c for c in df.columns if c.startswith(column_prefix)]

    # Single-pass collect_set to get all distinct values per column (1 Spark job)
    distinct_sets = df.agg(
        *[F.collect_set(c).alias(c) for c in indicator_cols]
    ).collect()[0].asDict()

    # Classify: subset of {"Sim", "Não"} → binary, otherwise categorical
    binary_cols, categorical_cols = [], []
    for col_name in indicator_cols:
        meaningful = set(str(v) for v in distinct_sets[col_name]) - set(NON_RESPONSE) - {"None", "nan"}
        if meaningful <= {"Sim", "Não"}:
            binary_cols.append(col_name)
        else:
            categorical_cols.append(col_name)

    print(f"Binary: {len(binary_cols)} | Categorical: {len(categorical_cols)}")
    print(f"Categorical: {categorical_cols}")

    # Build all transformations as a single select() (flat plan, no nested withColumn)
    select_exprs = [F.col("cod_munic")]
    for col_name in indicator_cols:
        if col_name in binary_cols:
            # Sim/Não → boolean, everything else → NULL
            select_exprs.append(
                F.when(F.col(col_name) == "Sim", True)
                .when(F.col(col_name) == "Não", False)
                .otherwise(None)
                .alias(col_name)
            )
        else:
            # Trim whitespace; non-response codes → NULL; otherwise keep value
            select_exprs.append(
                F.when(F.trim(F.col(col_name)).isin(NON_RESPONSE), None)
                .otherwise(F.trim(F.col(col_name)))
                .alias(col_name)
            )

    treated_df = df.select(*select_exprs)

    return {
        "df": treated_df,
        "binary_cols": binary_cols,
        "categorical_cols": categorical_cols,
        "meanings": meanings,
    }

#### Treating applying in indicators

In [0]:
# Apply the reusable treatment function to MTIC (Informática e Comunicação) indicators
# column_prefix="mtic" matches columns like mtic011, mtic04, mtic12a1, etc.
# dict_table_name=bronze_tbl_dict is the munic_raw_dict table defined in cell 4
result_info = treat_survey_indicators(
    df_info_indicators,
    column_prefix="mtic",
    dict_table_name=bronze_tbl_dict,
    ano_edicao=ano_edicao
)

# Unpack the results dict into individual variables
df_info_indicators = result_info["df"]                   # treated DataFrame (binary→boolean, etc.)
mtic_meanings = result_info["meanings"]                 # dict: {"mtic011": "Correio", ...}
binary_cols_info = result_info["binary_cols"]           # list of 188 binary column names
categorical_cols_info = result_info["categorical_cols"]  # list of 10 categorical column names

print(f"\nTreatment applied to {len(binary_cols_info) + len(categorical_cols_info)} MTIC columns")

# Verify the treatment: schema changes and null counts
print("=== Schema after treatment (first 15 MTIC columns) ===")
df_info_indicators.printSchema()

# Show null counts for a sample of binary and categorical columns
# This confirms that non-response codes (Recusa, Não informou, -, etc.) were converted to NULL
print("\n=== Null counts (non-response codes converted to NULL) ===")
sample_check = ["mtic011", "mtic04", "mtic10", "mtic11", "mtic15", "mtic29", "mtic31"]

# Count the rows ONCE, outside the loop. Calling .count() inside the loop would
# trigger one full Spark job per iteration just to recompute the same number.
total = df_info_indicators.count()

for col_name in sample_check:
    null_count = df_info_indicators.filter(F.col(col_name).isNull()).count()
    meaning = mtic_meanings.get(col_name, '?')[:50]
    print(f"  {col_name} ({meaning}): {null_count}/{total} nulls")

## Step 4 - Treat the survey indicators from "Governanca" worksheet

In [0]:
# Select only cod_munic and the MGOV indicator columns from the gov table
# Drop the city descriptor and metadata columns
mgov_cols = [c for c in df_raw_gov.columns if c.startswith("mgov")]
df_gov_indicators = df_raw_gov.select("cod_munic", *mgov_cols)

print(f"Gov indicators: {len(mgov_cols)} MGOV columns")
display(df_gov_indicators.limit(5))

In [0]:
# Apply the SAME reusable treatment function to MGOV (Governanca) indicators
# column_prefix="mgov" matches columns like mgov01, mgov011a, mgov36, etc.
# We reuse the same function — no code duplication between MTIC and MGOV treatment
result_gov = treat_survey_indicators(
    df_gov_indicators,
    column_prefix="mgov",
    dict_table_name=bronze_tbl_dict,
    ano_edicao=ano_edicao
)

# Unpack the results
df_gov_indicators = result_gov["df"]                    # treated DataFrame
mgov_meanings = result_gov["meanings"]                 # dict: {"mgov01": "...", ...}
binary_cols_gov = result_gov["binary_cols"]             # list of binary column names
categorical_cols_gov = result_gov["categorical_cols"]  # list of categorical column names

print(f"\nTreatment applied to {len(binary_cols_gov) + len(categorical_cols_gov)} MGOV columns")
display(df_gov_indicators.limit(5))

## Step 5 - Integrity Check
- primary key uniqueness and row count match (5,570 == 5,570)

In [0]:
# Check primary key uniqueness in both source tables
info_count = df_raw_info.count()
gov_count = df_raw_gov.count()

# .distinct() is REQUIRED here: .select() alone does not deduplicate, so without it
# these would just recount the rows and the uniqueness assertions below would always
# pass, no matter how many duplicate keys the table actually holds.
info_distinct = df_raw_info.select("cod_munic").distinct().count()
gov_distinct = df_raw_gov.select("cod_munic").distinct().count()

print(f"Info table: {info_count} rows, {info_distinct} distinct cod_munic")
print(f"Gov table:  {gov_count} rows, {gov_distinct} distinct cod_munic")

assert info_count == info_distinct, f"Duplicate cod_munic found in info table"
assert gov_count == gov_distinct, f"Duplicate cod_munic found in gov table"
assert info_count == gov_count, f"Row count mismatch: {info_count} != {gov_count}"
assert info_count == 5570, f"Expected 5.570 rows, got {info_count}"

print(f"\nAll integrity checks passed: {info_count} == {gov_count} == 5.570 rows")

## Step 6 - Merge data tables

In [0]:
# Merge the indicator tables on cod_munic (inner join)
df_merged_indicators = df_info_indicators.join(df_gov_indicators, on="cod_munic", how="inner")

# Cast cod_munic to string as cod_ibge for the join with city metainfo
df_merged_indicators = (
    df_merged_indicators
    .withColumn("cod_ibge", F.col("cod_munic").cast("string"))
    .drop("cod_munic")
)

# Join with city metainfo on cod_ibge
df_silver_data = df_city_metainfo.join(df_merged_indicators, on="cod_ibge", how="inner")

merged_count = df_silver_data.count()

print(f"Merged table row count: {merged_count}")
print(f"Merged table column count: {len(df_silver_data.columns)}")

# An inner join silently drops rows whose key is missing on either side.
# Assert the municipality count survived both joins instead of trusting the print.
assert merged_count == 5570, f"Rows lost in the merge: expected 5.570, got {merged_count}"

## Step 7 - Write to silver

In [0]:
# The indicator treatment in Steps 3 and 4 rebuilds the DataFrames with select(),
# which drops the lineage columns written by the bronze job. Re-attach them here so
# the silver tables still answer "where did this row come from and when".
#
# reference_year and munic_source_url come from env_variables (env notebook).

df_silver_data = df_silver_data.withColumns({
    "ano_edicao": F.lit(ano_edicao),
    "dataset_name": F.lit("IBGE_MUNIC"),
    "reference_year": F.lit(reference_year),
    "source_url": F.lit(munic_source_url),
    "source_file": F.lit(munic_file),
    "silver_processed_at": F.current_timestamp(),
})

# Place entity key first, then partition key: cod_ibge then ano_edicao
all_cols = df_silver_data.columns
other_cols = [c for c in all_cols if c not in ("ano_edicao", "cod_ibge")]
df_silver_data = df_silver_data.select("cod_ibge", "ano_edicao", *other_cols)

display(
    df_silver_data.select(
        "cod_ibge", "ano_edicao", "dataset_name", "reference_year", "silver_processed_at"
    ).limit(5)
)

In [0]:
# --- Write to silver using MERGE (upsert), partitioned by ano_edicao ---
# upsert_delta_table is imported from env_functions notebook

# Clean the dictionary - place cod_variable (entity key) first, then ano_edicao (partition key)
df_clean_dict = df_raw_dict.select("cod_variable", "ano_edicao", "meaning")

# Dictionary table - merge key: ano_edicao + cod_variable
upsert_delta_table(
    df_clean_dict,
    silver_tbl_dict,
    ["ano_edicao", "cod_variable"],
    ano_edicao,
    f"Cleaned MUNIC dictionary - {ano_edicao} edition"
)

# Data table - merge key: ano_edicao + cod_ibge
upsert_delta_table(
    df_silver_data,
    silver_tbl_data,
    ["ano_edicao", "cod_ibge"],
    ano_edicao,
    f"Cleaned and merged MUNIC survey data - {ano_edicao} edition"
)

print(f"\nDict rows: {df_clean_dict.count()}")
print(f"Data rows: {df_silver_data.count()}")

## Seing the History of the silvers_tables

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {silver_tbl_data}"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {silver_tbl_dict}"))